# ASCII Graph Rendering Reference

Developer-facing classes and functions defined in `langchain_core.runnables.graph_ascii`.


# `VertexViewer: object`

`VertexViewer` stores the width and height of a graph vertex box used by the ASCII graph layout engine.

## Class Field

```python
HEIGHT: int = 3 # Fixed vertex-box height containing top border, text, and bottom border
```

## Constructor

```python
VertexViewer(
    name: str, # Text displayed inside the vertex box
) -> None # Initialize the vertex dimensions
```

The width is calculated from the length of `name` plus space for the left and right borders.

## Properties

### `h`

Returns the height of the vertex box.

```python
h: int # Return the vertex-box height
```

### `w`

Returns the width of the vertex box.

```python
w: int # Return the vertex-box width
```


In [1]:
from langchain_core.runnables.graph_ascii import VertexViewer # Import VertexViewer

vertex_name: str = "Process Data" # Define the text displayed inside the vertex box

viewer: VertexViewer = VertexViewer(vertex_name) # Create the VertexViewer object

height: int = viewer.h # Get the fixed vertex-box height

width: int = viewer.w # Get the calculated vertex-box width

print("Vertex name:", vertex_name) # Display the vertex name

print("Height:", height) # Display the box height

print("Width:", width) # Display the box width

Vertex name: Process Data
Height: 3
Width: 14


# `AsciiCanvas: object`

`AsciiCanvas` provides a two-dimensional character canvas for drawing ASCII points, lines, text, and boxes.

## Class Field

```python
TIMEOUT: int = 10 # Timeout value associated with ASCII rendering
```

## Fields

```python
cols: int # Number of columns in the canvas
lines: int # Number of rows in the canvas
canvas: list[list[str]] # Two-dimensional character grid
```

## Constructor

```python
AsciiCanvas(
    cols: int, # Number of canvas columns greater than 1
    lines: int, # Number of canvas rows greater than 1
) -> None # Initialize the blank ASCII canvas
```

Raises `ValueError` when either dimension is less than or equal to `1`.

## Methods

### `draw`

Converts the character grid into one multiline string.

```python
draw(
    self, # Current ASCII canvas
) -> str # Return the rendered ASCII canvas
```

### `point`

Places one character at the specified coordinates.

```python
point(
    self, # Current ASCII canvas
    x: int, # Horizontal coordinate from 0 to cols minus 1
    y: int, # Vertical coordinate from 0 to lines minus 1
    char: str, # Single character placed on the canvas
) -> None # Update the selected canvas position
```

Raises `ValueError` when `char` contains more than one character or when the coordinates are outside the canvas.

### `line`

Draws a straight approximated line between two coordinates.

```python
line(
    self, # Current ASCII canvas
    x0: int, # Horizontal coordinate of the starting point
    y0: int, # Vertical coordinate of the starting point
    x1: int, # Horizontal coordinate of the ending point
    y1: int, # Vertical coordinate of the ending point
    char: str, # Character used to draw the line
) -> None # Draw the line on the canvas
```

### `text`

Writes a string horizontally from the supplied coordinates.

```python
text(
    self, # Current ASCII canvas
    x: int, # Horizontal coordinate where the text begins
    y: int, # Vertical coordinate where the text begins
    text: str, # Text written on the canvas
) -> None # Write the text onto the canvas
```

### `box`

Draws a rectangular ASCII box using `+`, `-`, and `|`.

```python
box(
    self, # Current ASCII canvas
    x0: int, # Horizontal coordinate of the top-left corner
    y0: int, # Vertical coordinate of the top-left corner
    width: int, # Box width greater than 1
    height: int, # Box height greater than 1
) -> None # Draw the box on the canvas
```

Raises `ValueError` when the width or height is less than or equal to `1`.

In [3]:
from langchain_core.runnables.graph_ascii import AsciiCanvas # Import AsciiCanvas

canvas: AsciiCanvas = AsciiCanvas( # Create a blank ASCII canvas
    cols=35, # Set the canvas width
    lines=12, # Set the canvas height
) # Finish creating the canvas

canvas.box( # Draw the first rectangular box
    x0=1, # Set the left coordinate
    y0=1, # Set the top coordinate
    width=12, # Set the box width
    height=5, # Set the box height
) # Finish drawing the first box

canvas.text( # Write text inside the first box
    x=4, # Set the text horizontal position
    y=3, # Set the text vertical position
    text="Input", # Set the displayed text
) # Finish writing the first label

canvas.box( # Draw the second rectangular box
    x0=21, # Set the left coordinate
    y0=6, # Set the top coordinate
    width=12, # Set the box width
    height=5, # Set the box height
) # Finish drawing the second box

canvas.text( # Write text inside the second box
    x=23, # Set the text horizontal position
    y=8, # Set the text vertical position
    text="Output", # Set the displayed text
) # Finish writing the second label

canvas.line( # Draw a line connecting both boxes
    x0=13, # Set the line starting x-coordinate
    y0=3, # Set the line starting y-coordinate
    x1=20, # Set the line ending x-coordinate
    y1=8, # Set the line ending y-coordinate
    char="*", # Use an asterisk to draw the line
) # Finish drawing the line

canvas.point( # Place one individual character
    x=17, # Set the point horizontal position
    y=1, # Set the point vertical position
    char="X", # Set the displayed character
) # Finish placing the point

result: str = canvas.draw() # Convert the canvas into a multiline string

print(result) # Display the completed ASCII drawing

                                   
 +----------+    X                 
 |          |                      
 |  Input   |*                     
 |          | **                   
 +----------+   *                  
                 *   +----------+  
                  ** |          |  
                    *| Output   |  
                     |          |  
                     +----------+  
                                   


# `draw_ascii`

Builds a directed acyclic graph layout and returns its ASCII representation.

```python
draw_ascii(
    vertices: Mapping[str, str], # Vertex identifiers mapped to display labels
    edges: Sequence[LangEdge], # Directed graph edges connecting the vertices
) -> str # Return the rendered multiline ASCII graph
```

## Behaviour

- Uses the Sugiyama layout algorithm supplied by `grandalf`.
- Draws edges before vertices so that vertex boxes overwrite intersecting edge characters.
- Uses `*` for normal edges.
- Uses `.` for conditional edges.
- Automatically calculates and shifts graph coordinates into a positive canvas area.
- Raises `ImportError` when `grandalf` is not installed.
- Raises `ValueError` when canvas dimensions or calculated edge coordinates are invalid.

## Required Optional Dependency

```bash
pip install grandalf
```

In [2]:

from langchain_core.runnables.graph import Edge # Import the graph Edge type
from langchain_core.runnables.graph_ascii import draw_ascii # Import the ASCII graph function

vertices: dict[str, str] = { # Create node IDs mapped to displayed labels
    "start": "Start", # Define the starting node
    "check": "Check Value", # Define the condition-checking node
    "positive": "Positive", # Define the positive-result node
    "default": "Zero or Negative", # Define the default-result node
} # Finish defining the vertices

edges: list[Edge] = [ # Create the directed connections between nodes
    Edge( # Create the first normal edge
        source="start", # Set Start as the source node
        target="check", # Set Check Value as the target node
    ), # Finish creating the first edge
    Edge( # Create the conditional positive edge
        source="check", # Set Check Value as the source node
        target="positive", # Set Positive as the target node
        data="value > 0", # Store a description for the edge
        conditional=True, # Draw this edge using dots
    ), # Finish creating the conditional edge
    Edge( # Create the conditional default edge
        source="check", # Set Check Value as the source node
        target="default", # Set Zero or Negative as the target node
        data="otherwise", # Store a description for the edge
        conditional=True, # Draw this edge using dots
    ), # Finish creating the default edge
] # Finish defining the edges

ascii_graph: str = draw_ascii( # Generate the ASCII graph
    vertices=vertices, # Supply the graph vertices
    edges=edges, # Supply the graph edges
) # Finish generating the graph

print(ascii_graph) # Display the ASCII graph

              +-------+                    
              | Start |                    
              +-------+                    
                  *                        
                  *                        
                  *                        
           +-------------+                 
           | Check Value |                 
           +-------------+                 
           ...          ...                
          .                .               
        ..                  ..             
+----------+         +------------------+  
| Positive |         | Zero or Negative |  
+----------+         +------------------+  
